In [ ]:
from google.colab import files
import pandas as pd

# ── 1. Upload files one at a time ─────────────────────────────────────────────
print("Step 1: Upload demographics_pa.csv")
files.upload()

print("Step 2: Upload philly_tract_centroids.csv")
files.upload()

# ── 2. Load ───────────────────────────────────────────────────────────────────
demo      = pd.read_csv("demographics_pa.csv", dtype={"GEOID": str})
centroids = pd.read_csv("philly_tract_centroids.csv", dtype={"GEOID10": str})

# ── 3. Filter to 2023 (one row per tract) ─────────────────────────────────────
demo2023 = demo[demo["year"] == 2023].copy()

# ── 4. Compute majority race (>50% threshold) ─────────────────────────────────
def majority_race(row):
    races = {
        "White":    row["pct_white"],
        "Black":    row["pct_black"],
        "Asian":    row["pct_asian"],
        "Hispanic": row["pct_hispanic"],
    }
    top = max(races, key=races.get)
    return top if races[top] > 50 else "None"

demo2023["majority_race"] = demo2023.apply(majority_race, axis=1)

# ── 5. Merge with centroids ───────────────────────────────────────────────────
demo2023["GEOID"]        = demo2023["GEOID"].str.zfill(11)
centroids["GEOID10"]     = centroids["GEOID10"].str.zfill(11)

merged = demo2023.merge(
    centroids[["GEOID10", "lat", "lon"]],
    left_on="GEOID",
    right_on="GEOID10",
    how="inner"
)

print(f"Total tracts: {len(merged)}")
print(merged["majority_race"].value_counts())

# ── 6. Keep useful columns ────────────────────────────────────────────────────
merged = merged[[
    "GEOID",
    "tract",
    "lat",
    "lon",
    "majority_race",
    "pct_white",
    "pct_black",
    "pct_asian",
    "pct_hispanic",
    "median_income",
    "total_population",
]].copy()

# ── 7. Save & download ────────────────────────────────────────────────────────
merged.to_csv("all_tracts_demo.csv", index=False)
files.download("all_tracts_demo.csv")
print("Done! all_tracts_demo.csv downloaded.")


Step 1: Upload demographics_pa.csv


Saving demographics_pa.csv to demographics_pa.csv
Step 2: Upload philly_tract_centroids.csv


Saving philly_tract_centroids.csv to philly_tract_centroids.csv
Total tracts: 363
majority_race
Black       134
White       132
None         77
Hispanic     20
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! all_tracts_demo.csv downloaded.
